In [1]:
import pandas as pd

# Load both sheets from the Excel file
df1 = pd.read_excel("online_retail_II.xlsx", sheet_name="Year 2009-2010")
df2 = pd.read_excel("online_retail_II.xlsx", sheet_name="Year 2010-2011")

# Combine both sheets into one big table
df = pd.concat([df1, df2], ignore_index=True)

# Check how many rows we have
print(df.shape)
print(df.columns)

(1067371, 8)
Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')


In [2]:
# See the first 5 rows
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
# See data types and how many non-null values exist
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


In [4]:
# See basic statistics (min, max, average etc.)
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,1.067371e+06,1067371,1.067371e+06,824364.000000
mean,9.938898e+00,2011-01-02 21:13:55.394028544,4.649388e+00,15324.638504
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000
50%,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000
75%,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.727058e+02,NaN,1.235531e+02,1697.464450


In [5]:
# Count how many missing values are in each column
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [6]:
# Problem 1 — Remove cancellations
# Invoices starting with "C" mean the customer cancelled/returned
df = df[~df["Invoice"].astype(str).str.startswith("C")]
print("After removing cancellations:", df.shape)

After removing cancellations: (1047877, 8)


In [7]:
# Problem 2 — Remove rows where Customer ID or Description is missing
# We can't analyse customers without a Customer ID
df.dropna(subset=["Customer ID", "Description"], inplace=True)
print("After dropping nulls:", df.shape)

After dropping nulls: (805620, 8)


In [8]:
# Problem 3 — Fix data types
# InvoiceDate should be a date, not plain text
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# Customer ID should be a whole number, not a decimal
df["Customer ID"] = df["Customer ID"].astype(int)

In [9]:
# Problem 4 — Remove rows with bad Quantity or Price
# Negative or zero values don't make sense for valid sales
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]
print("After removing bad rows:", df.shape)

After removing bad rows: (805549, 8)


In [10]:
# Problem 5 — Remove duplicates
df.drop_duplicates(inplace=True)
print("After removing duplicates:", df.shape)

After removing duplicates: (779425, 8)


In [11]:
# Revenue = how much money each row made
df["Revenue"] = df["Quantity"] * df["Price"]

# Extract time parts from the date
df["Year"]      = df["InvoiceDate"].dt.year
df["Month"]     = df["InvoiceDate"].dt.month
df["YearMonth"] = df["InvoiceDate"].dt.to_period("M").astype(str)

# Extract category from StockCode
# StockCode letters at the start roughly indicate product category
df["Category"] = df["StockCode"].astype(str).str.extract(r"^([A-Za-z]+)")
df["Category"] = df["Category"].fillna("General")

print(df.head())

  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          InvoiceDate  Price  Customer ID         Country  Revenue  Year  \
0 2009-12-01 07:45:00   6.95        13085  United Kingdom     83.4  2009   
1 2009-12-01 07:45:00   6.75        13085  United Kingdom     81.0  2009   
2 2009-12-01 07:45:00   6.75        13085  United Kingdom     81.0  2009   
3 2009-12-01 07:45:00   2.10        13085  United Kingdom    100.8  2009   
4 2009-12-01 07:45:00   1.25        13085  United Kingdom     30.0  2009   

   Month YearMonth Category  
0     12   2009-12  General  
1     12   2009-12  General  
2     12   2009-12  General  
3 

In [12]:
# Save as CSV — this is your clean dataset
df.to_csv("retail_clean.csv", index=False)
print("Clean data saved. Rows:", len(df))

Clean data saved. Rows: 779425


In [13]:
import sqlite3

# Create a database file called retail.db
conn = sqlite3.connect("retail.db")

# Load your clean dataframe into the database as a table
df.to_sql("transactions", conn, if_exists="replace", index=False)

print("Data loaded into SQLite successfully!")

Data loaded into SQLite successfully!


In [14]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("retail.db")

# Always use this function to run a query and get results back as a table
def run_query(sql):
    return pd.read_sql(sql, conn)

In [15]:
sales_trends = run_query("""
    SELECT YearMonth,
           ROUND(SUM(Revenue), 2)        AS total_revenue,
           COUNT(DISTINCT Invoice)        AS total_orders,
           COUNT(DISTINCT "Customer ID") AS unique_customers
    FROM transactions
    GROUP BY YearMonth
    ORDER BY YearMonth
""")

print(sales_trends)
sales_trends.to_csv("sales_trends.csv", index=False)

   YearMonth  total_revenue  total_orders  unique_customers
0    2009-12      683504.01          1512               955
1    2010-01      555802.67          1011               720
2    2010-02      504558.96          1104               772
3    2010-03      696978.47          1524              1057
4    2010-04      591982.00          1329               942
5    2010-05      597833.38          1377               966
6    2010-06      636371.13          1497              1041
7    2010-07      589736.17          1381               928
8    2010-08      602224.60          1293               911
9    2010-09      829013.95          1689              1145
10   2010-10     1033112.01          2133              1497
11   2010-11     1166460.02          2587              1607
12   2010-12      570422.73          1400               885
13   2011-01      568101.31           987               741
14   2011-02      446084.92           997               758
15   2011-03      594081.76          132

In [16]:
regional = run_query("""
    SELECT Country,
           ROUND(SUM(Revenue), 2)         AS total_revenue,
           COUNT(DISTINCT "Customer ID")  AS unique_customers,
           COUNT(DISTINCT Invoice)        AS total_orders
    FROM transactions
    GROUP BY Country
    ORDER BY total_revenue DESC
""")

print(regional)
regional.to_csv("regional_distribution.csv", index=False)

                 Country  total_revenue  unique_customers  total_orders
0         United Kingdom    14389234.92              5350         33541
1                   EIRE      616570.54                 5           567
2            Netherlands      554038.09                22           228
3                Germany      425019.71               107           789
4                 France      348768.96                95           614
5              Australia      169283.46                15            95
6                  Spain      108332.49                41           154
7            Switzerland      100061.94                22            90
8                 Sweden       91515.82                19           104
9                Denmark       68580.69                12            43
10               Belgium       65387.82                29           149
11                Norway       56322.50                13            45
12              Portugal       55554.78                24       

In [17]:
retention = run_query("""
    WITH first_purchase AS (
        SELECT "Customer ID",
               MIN(YearMonth) AS cohort_month
        FROM transactions
        GROUP BY "Customer ID"
    )
    SELECT f.cohort_month,
           t.YearMonth                        AS purchase_month,
           COUNT(DISTINCT t."Customer ID")    AS retained_customers
    FROM transactions t
    JOIN first_purchase f
      ON t."Customer ID" = f."Customer ID"
    GROUP BY f.cohort_month, t.YearMonth
    ORDER BY cohort_month, purchase_month
""")

print(retention)
retention.to_csv("customer_retention.csv", index=False)

    cohort_month purchase_month  retained_customers
0        2009-12        2009-12                 955
1        2009-12        2010-01                 337
2        2009-12        2010-02                 319
3        2009-12        2010-03                 406
4        2009-12        2010-04                 363
..           ...            ...                 ...
320      2011-10        2011-11                  71
321      2011-10        2011-12                  35
322      2011-11        2011-11                 191
323      2011-11        2011-12                  27
324      2011-12        2011-12                  28

[325 rows x 3 columns]


In [18]:
top_products = run_query("""
    SELECT Description,
           SUM(Quantity)          AS units_sold,
           ROUND(SUM(Revenue), 2) AS total_revenue
    FROM transactions
    GROUP BY Description
    ORDER BY total_revenue DESC
    LIMIT 20
""")

print(top_products)
top_products.to_csv("top_products.csv", index=False)

                            Description  units_sold  total_revenue
0              REGENCY CAKESTAND 3 TIER       24124      277656.25
1    WHITE HANGING HEART T-LIGHT HOLDER       91757      247048.01
2           PAPER CRAFT , LITTLE BIRDIE       80995      168469.60
3                                Manual        9384      151777.67
4               JUMBO BAG RED RETROSPOT       74224      134307.44
5                               POSTAGE        5235      124648.04
6         ASSORTED COLOUR BIRD ORNAMENT       78234      124351.86
7                         PARTY BUNTING       23460      103283.38
8        MEDIUM CERAMIC TOP STORAGE JAR       77916       81416.73
9       PAPER CHAIN KIT 50'S CHRISTMAS        28380       76598.18
10                        CHILLI LIGHTS       14843       69084.30
11                 JUMBO BAG STRAWBERRY       35842       64127.77
12             BLACK RECORD COVER FRAME       18417       63092.12
13  ROTATING SILVER ANGELS T-LIGHT HLDR       28606       5588

In [19]:
df = pd.read_csv("retail_clean.csv", parse_dates=["InvoiceDate"])

# Set snapshot date as the day after the last transaction
snapshot = df["InvoiceDate"].max()

# Calculate R, F, M for each customer
rfm = df.groupby("Customer ID").agg(
    Recency   = ("InvoiceDate", lambda x: (snapshot - x.max()).days),
    Frequency = ("Invoice",     "nunique"),
    Monetary  = ("Revenue",     "sum")
).reset_index()

# Score each metric 1 to 3
# Recency: lower days = better = score 3
rfm["R_Score"] = pd.qcut(rfm["Recency"],   q=3, labels=[3,2,1]).astype(int)
rfm["F_Score"] = pd.qcut(rfm["Frequency"], q=3, labels=[1,2,3]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["Monetary"],  q=3, labels=[1,2,3]).astype(int)

# Add scores together
rfm["RFM_Total"] = rfm["R_Score"] + rfm["F_Score"] + rfm["M_Score"]

# Label segments based on total score
rfm["Segment"] = pd.cut(rfm["RFM_Total"],
                         bins=[0, 4, 6, 9],
                         labels=["Low Value", "Mid Value", "High Value"])

rfm.to_csv("customer_segments.csv", index=False)
print(rfm["Segment"].value_counts())

Segment
High Value    2375
Low Value     1912
Mid Value     1591
Name: count, dtype: int64


In [20]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.chart import BarChart, Reference
from openpyxl.styles import Font, PatternFill, Alignment

df = pd.read_csv("retail_clean.csv")

# Create discount tiers based on price
# Lower price = heavier discount applied originally
df["Discount Tier"] = pd.cut(
    df["Price"],
    bins=[0, 1, 3, 10, 50, 9999],
    labels=["Heavy Discount", "High Discount",
            "Medium Discount", "Low Discount", "No Discount"]
)

# Build pivot table
pivot = df.pivot_table(
    index="Discount Tier",
    values=["Revenue", "Quantity"],
    aggfunc="sum"
).reset_index()

pivot["Revenue"] = pivot["Revenue"].round(2)

# Create Excel workbook
wb = Workbook()
ws = wb.active
ws.title = "Discount Impact"

# Write data into sheet
for r in dataframe_to_rows(pivot, index=False, header=True):
    ws.append(r)

# Style the header row
for cell in ws[1]:
    cell.font      = Font(bold=True, color="FFFFFF")
    cell.fill      = PatternFill("solid", fgColor="4472C4")
    cell.alignment = Alignment(horizontal="center")

# Auto-width columns
for col in ws.columns:
    max_len = max(len(str(cell.value or "")) for cell in col)
    ws.column_dimensions[col[0].column_letter].width = max_len + 4

# Add bar chart
chart      = BarChart()
chart.title        = "Revenue Impact by Discount Tier"
chart.y_axis.title = "Total Revenue (£)"
chart.x_axis.title = "Discount Tier"

data = Reference(ws, min_col=2, min_row=1, max_row=ws.max_row)
cats = Reference(ws, min_col=1, min_row=2, max_row=ws.max_row)
chart.add_data(data, titles_from_data=True)
chart.set_categories(cats)
ws.add_chart(chart, "F2")

wb.save("discount_impact_report.xlsx")
print("Excel report saved!")

Excel report saved!


C:\Users\dellf\AppData\Local\Temp\ipykernel_22800\1077203110.py:19: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = df.pivot_table(
